# Notebook 3: PPO Fine-Tuning

## 3.0 Preamble

This notebook fine-tunes the behaviour-cloning (BC) policy from Notebook 02 using Proximal Policy Optimisation (PPO) through live interaction with the CARLA simulator. PPO refines the imitation-learned policy by maximising a shaped reward signal that captures safe, comfortable, and efficient driving. The training introduces three distinct driving style variants -- chill, standard, and hurry -- each governed by a different reward profile that reshapes the reward landscape, similar to the comfort modes found in modern Tesla and BMW vehicles. Each town-style combination produces its own checkpoint (`ppo_{town}_{style}_best.pt`), yielding up to 18 fine-tuned policies across the six training towns.

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Ensure project root is on sys.path so src.* imports resolve
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.agents.ppo_agent import PPOAgent

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

## 3.1 Configuration

### 3.1.1 Loading Parameters

All PPO hyperparameters are stored in `configs/ppo.yaml` and loaded via `load_config("ppo")`. This ensures that the notebook, the PPOAgent, and any future scripts all read from a single source of truth. Hardcoding hyperparameters anywhere else is deliberately avoided so that a config change propagates everywhere automatically. The key values printed below control the learning rate, clipping range, rollout length, and curriculum schedule.

In [ ]:
cfg = load_config("ppo")

print("=== PPO Hyperparameters ===")
hyper_keys = [
    "seed", "dropout", "lr", "clip_eps", "entropy_coef",
    "value_loss_coef", "n_steps", "batch_size", "n_epochs_ppo",
    "gamma", "gae_lambda", "total_timesteps",
    "curriculum_switch_step", "max_grad_norm",
    "cam_w", "cam_h", "crop_top", "crop_bottom",
]
for k in hyper_keys:
    print(f"  {k:30s} = {cfg[k]}")

print(f"\n=== Curriculum Weather Phases ===")
print(f"  Phase 1 (steps < {cfg['curriculum_switch_step']}): {cfg['weather_phase1']}")
print(f"  Phase 2 (steps >= {cfg['curriculum_switch_step']}): {cfg['weather_phase2']}")

### 3.1.2 Reward Profile Comparison

The three driving styles are inspired by the comfort and sport modes found in consumer vehicles such as Tesla's Chill, Standard, and Sport settings. Each style adjusts the weights on three reward shaping components -- jerk penalty, speed bonus, and lane-change penalty -- to encourage qualitatively different driving behaviour. A "chill" driver is penalised heavily for jerky acceleration and lane changes but receives little reward for going fast, whereas a "hurry" driver is rewarded strongly for maintaining high speed with relaxed smoothness constraints. The "standard" profile provides a balanced middle ground.

In [ ]:
profiles = cfg["reward_profiles"]

df_profiles = pd.DataFrame(profiles).T
df_profiles.index.name = "style"
df_profiles = df_profiles[["jerk_penalty", "speed_bonus", "lane_change_penalty"]]
print(df_profiles.to_string())

## 3.2 Environment Setup

### 3.2.1 CARLA Connection Verification

Before launching any training, we verify that the CARLA simulator is reachable on `localhost:2000`. The server must already be running with the correct town loaded via the `-dx12` RHI flag, which is mandatory for the RTX 5080 Blackwell GPU to avoid camera sensor deadlocks. If the connection fails here, check that `scripts/launch_carla.bat` was executed with the correct map argument. Runtime map switching via `client.load_world()` is not used because it causes a Vulkan null-pointer crash on this hardware.

In [ ]:
try:
    import carla
    client = carla.Client("localhost", 2000)
    client.set_timeout(10.0)
    server_version = client.get_server_version()
    world = client.get_world()
    current_map = world.get_map().name
    print(f"CARLA server version : {server_version}")
    print(f"Currently loaded map : {current_map}")
except Exception as e:
    print(f"CARLA connection failed: {e}")
    print("Start CARLA first:  CarlaUE4-Win64-Shipping.exe -dx12 /Game/Carla/Maps/Town01")

### 3.2.2 BC Warm Start Verification

PPO does not train from scratch -- it starts from the BC-initialised weights produced in Notebook 02. This "warm start" gives the policy a reasonable starting behaviour so that early CARLA episodes are survivable enough for PPO to collect useful reward signals. Without BC initialisation, the randomly initialised policy would crash almost immediately, producing near-zero reward and making credit assignment extremely difficult. Here we verify that the BC checkpoint exists and report its parameter count.

In [ ]:
bc_path = PROJECT_ROOT / "models" / "BC_model_best.pt"
assert bc_path.exists(), f"BC checkpoint not found: {bc_path}"

bc_state_dict = torch.load(bc_path, map_location="cpu")
total_params = sum(p.numel() for p in bc_state_dict.values())
print(f"BC checkpoint        : {bc_path.name}")
print(f"File size            : {bc_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Total parameters     : {total_params:,}")
print(f"Layers               : {len(bc_state_dict)}")
print("BC weights loaded successfully -- ready for PPO warm start.")

## 3.3 Reward Function

### 3.3.1 Reward Components

The PPO reward signal is composed of four additive components, each targeting a different aspect of driving quality. The base reward comes directly from CARLA's environment (progress toward the destination minus collision penalties). The three shaping terms -- jerk penalty, speed bonus, and lane-change penalty -- are multiplied by style-specific weights to produce qualitatively different driving personalities. The jerk penalty uses a second-order finite difference of speed to penalise abrupt acceleration changes, promoting smoother rides.

| Component | Formula | Purpose |
|---|---|---|
| `base_reward` | CARLA env reward (progress - collision) | Forward progress toward destination |
| `jerk_penalty` | `w_jerk * abs(a_t - 2*a_{t-1} + a_{t-2}) / dt^2` | Penalise abrupt acceleration changes |
| `speed_bonus` | `w_speed * (speed_kmh / speed_limit)` | Reward maintaining appropriate speed |
| `lane_change_penalty` | `w_lane * (lane_id != prev_lane_id)` | Penalise unnecessary lane changes |

### 3.3.2 Style Profiles

Each style profile applies a different set of weights to the three shaping terms described above, creating a unique reward landscape that encourages distinct driving behaviour. The chill profile strongly penalises jerk and lane changes, producing a smooth, conservative driver. The hurry profile rewards speed aggressively while relaxing smoothness constraints. The standard profile provides a balanced middle ground. These profiles are loaded from `configs/ppo.yaml` and passed to `compute_style_reward()` at every environment step during training.

In [ ]:
print("=== Reward Profiles (side-by-side) ===")
print(f"{'Component':<25s} {'chill':>8s} {'standard':>10s} {'hurry':>8s}")
print("-" * 55)
for component in ["jerk_penalty", "speed_bonus", "lane_change_penalty"]:
    vals = [profiles[s][component] for s in ["chill", "standard", "hurry"]]
    print(f"{component:<25s} {vals[0]:>8.1f} {vals[1]:>10.1f} {vals[2]:>8.1f}")

## 3.4 Training -- Chill Style

### 3.4.1 Per-Town Instructions

Due to a Vulkan null-pointer crash on the RTX 5080 Blackwell GPU, the CARLA simulator cannot switch maps at runtime via `client.load_world()`. Each town must be trained in a separate CARLA session. Before running each cell below, restart the CARLA server with the appropriate map loaded using the `-dx12` flag. The cells are independent -- if one town fails, the others are unaffected. Run them one at a time, restarting CARLA between each.

In [ ]:
TOWNS = ["Town01", "Town02", "Town03", "Town04", "Town05", "Town10HD"]

print("=== CARLA Launch Commands (run before each cell) ===")
for town in TOWNS:
    print(f"  {town}: CarlaUE4-Win64-Shipping.exe -dx12 /Game/Carla/Maps/{town}")

### 3.4.2 Run PPO Chill

Each cell below launches a full PPO training run for one town with the "chill" driving style. The agent collects rollouts in CARLA, computes style-shaped rewards with high jerk and lane-change penalties, and updates the policy for the configured number of timesteps. Checkpoints are saved as `ppo_{town}_chill_best.pt` whenever the rolling mean episode reward improves. Restart CARLA with the correct town before running each cell.

In [ ]:
# Town01 -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town01", style="chill")
history_town01_chill = agent.run()

In [ ]:
# Town02 -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town02", style="chill")
history_town02_chill = agent.run()

In [ ]:
# Town03 -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town03", style="chill")
history_town03_chill = agent.run()

In [ ]:
# Town04 -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town04", style="chill")
history_town04_chill = agent.run()

In [ ]:
# Town05 -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town05", style="chill")
history_town05_chill = agent.run()

In [ ]:
# Town10HD -- Chill
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town10HD", style="chill")
history_town10hd_chill = agent.run()

## 3.5 Training -- Standard Style

### 3.5.1 Per-Town Instructions

The standard style uses balanced reward weights (jerk=1.0, speed=1.0, lane_change=1.0). This is the default profile and represents a compromise between smoothness and urgency. As with the chill style, CARLA must be restarted with the correct town loaded before each cell. The same BC checkpoint is used as the warm start for all styles -- each style diverges from the same starting point, making their learned differences purely attributable to the reward shaping.

### 3.5.2 Run PPO Standard

Each cell launches PPO training with the "standard" driving style for one town. The balanced reward profile weights jerk, speed, and lane-change penalties equally, producing a policy that is neither overly cautious nor overly aggressive. Checkpoints are saved as `ppo_{town}_standard_best.pt`. Restart CARLA with the correct town before running each cell.

In [ ]:
# Town01 -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town01", style="standard")
history_town01_standard = agent.run()

In [ ]:
# Town02 -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town02", style="standard")
history_town02_standard = agent.run()

In [ ]:
# Town03 -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town03", style="standard")
history_town03_standard = agent.run()

In [ ]:
# Town04 -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town04", style="standard")
history_town04_standard = agent.run()

In [ ]:
# Town05 -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town05", style="standard")
history_town05_standard = agent.run()

In [ ]:
# Town10HD -- Standard
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town10HD", style="standard")
history_town10hd_standard = agent.run()

## 3.6 Training -- Hurry Style

### 3.6.1 Per-Town Instructions

The hurry style uses aggressive reward weights (jerk=0.5, speed=2.0, lane_change=0.5) that strongly incentivise maintaining high speed while relaxing penalties for abrupt manoeuvres and lane changes. This produces a policy that prioritises time efficiency over passenger comfort. As before, restart CARLA with the appropriate town before each cell. The hurry style is expected to achieve higher average speeds but may show more collisions and lane invasions compared to chill and standard during evaluation.

### 3.6.2 Run PPO Hurry

Each cell launches PPO training with the "hurry" driving style for one town. The speed-aggressive reward profile rewards high velocity while relaxing smoothness constraints. Checkpoints are saved as `ppo_{town}_hurry_best.pt`. Restart CARLA with the correct town before running each cell.

In [ ]:
# Town01 -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town01", style="hurry")
history_town01_hurry = agent.run()

In [ ]:
# Town02 -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town02", style="hurry")
history_town02_hurry = agent.run()

In [ ]:
# Town03 -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town03", style="hurry")
history_town03_hurry = agent.run()

In [ ]:
# Town04 -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town04", style="hurry")
history_town04_hurry = agent.run()

In [ ]:
# Town05 -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town05", style="hurry")
history_town05_hurry = agent.run()

In [ ]:
# Town10HD -- Hurry
agent = PPOAgent(bc_checkpoint="models/BC_model_best.pt", town="Town10HD", style="hurry")
history_town10hd_hurry = agent.run()

## 3.7 Training Diagnostics

### 3.7.1 Policy Loss Curves

The policy (surrogate) loss measures how much the updated policy deviates from the old policy within the PPO clipping region. A healthy training run shows the policy loss starting negative (the policy improves over the BC initialisation) and stabilising as the policy converges. We overlay all three driving styles per town to visualise whether any style is systematically harder to optimise. Large spikes may indicate the curriculum switch at step 50,000 when new weather conditions are introduced.

In [ ]:
TOWNS = ["Town01", "Town02", "Town03", "Town04", "Town05", "Town10HD"]
STYLES = ["chill", "standard", "hurry"]
STYLE_COLORS = {"chill": "#2196F3", "standard": "#4CAF50", "hurry": "#F44336"}
results_dir = PROJECT_ROOT / "results"

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes = axes.flatten()

for idx, town in enumerate(TOWNS):
    ax = axes[idx]
    for style in STYLES:
        hist_path = results_dir / f"ppo_{town}_{style}_training_history.json"
        if hist_path.exists():
            with open(hist_path) as f:
                hist = json.load(f)
            ax.plot(hist["policy_losses"], label=style, color=STYLE_COLORS[style], alpha=0.8)
    ax.set_title(town)
    ax.set_ylabel("Policy Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Policy Loss Curves -- All Styles per Town", fontsize=14)
plt.tight_layout()
plt.show()

### 3.7.2 Value Loss Curves

The value loss measures how accurately the critic network predicts future cumulative reward. A decreasing value loss indicates that the critic is learning the reward landscape successfully. Different driving styles have different reward scales (the hurry style's speed bonus inflates raw rewards), so absolute value-loss magnitudes are not directly comparable across styles. What matters is the downward trend within each style. A flat or increasing value loss may indicate that the reward signal is too noisy for the critic to model.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes = axes.flatten()

for idx, town in enumerate(TOWNS):
    ax = axes[idx]
    for style in STYLES:
        hist_path = results_dir / f"ppo_{town}_{style}_training_history.json"
        if hist_path.exists():
            with open(hist_path) as f:
                hist = json.load(f)
            ax.plot(hist["value_losses"], label=style, color=STYLE_COLORS[style], alpha=0.8)
    ax.set_title(town)
    ax.set_ylabel("Value Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Value Loss Curves -- All Styles per Town", fontsize=14)
plt.tight_layout()
plt.show()

### 3.7.3 Episode Reward Curves

Episode reward is the most intuitive diagnostic: it tracks the total shaped reward accumulated in each CARLA episode. An upward trend indicates that the policy is learning to drive better according to its style-specific reward function. The vertical dashed line marks the curriculum switch at step 50,000, where the weather pool expands from ClearNoon-only to all six weather presets. A temporary dip at this point is expected as the policy encounters unfamiliar visual conditions for the first time.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes = axes.flatten()

for idx, town in enumerate(TOWNS):
    ax = axes[idx]
    for style in STYLES:
        hist_path = results_dir / f"ppo_{town}_{style}_training_history.json"
        if hist_path.exists():
            with open(hist_path) as f:
                hist = json.load(f)
            rewards = hist["episode_rewards"]
            ax.plot(rewards, label=style, color=STYLE_COLORS[style], alpha=0.4, linewidth=0.8)
            # Rolling mean for clarity
            if len(rewards) > 10:
                rolling = np.convolve(rewards, np.ones(10) / 10, mode="valid")
                ax.plot(range(9, 9 + len(rolling)), rolling, color=STYLE_COLORS[style],
                        linewidth=2, alpha=0.9)
    # Annotate curriculum switch (approximate episode index)
    approx_ep = cfg["curriculum_switch_step"] // cfg["n_steps"]
    ax.axvline(x=approx_ep, color="gray", linestyle="--", alpha=0.5, label="curriculum switch")
    ax.set_title(town)
    ax.set_ylabel("Episode Reward")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Episode Reward Curves (with 10-episode rolling mean)", fontsize=14)
plt.tight_layout()
plt.show()

### 3.7.4 Entropy Curves

The entropy of the policy distribution measures the degree of exploration versus exploitation. High entropy means the policy is still exploring a wide range of actions; low entropy means it has committed to a narrow set of preferred actions. Entropy should decrease gradually during training as the policy specialises, but it should not collapse to zero -- that would indicate premature convergence to a deterministic policy that cannot adapt to new situations. The entropy coefficient in `configs/ppo.yaml` (currently 0.01) provides a regularisation floor to prevent this collapse.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes = axes.flatten()

for idx, town in enumerate(TOWNS):
    ax = axes[idx]
    for style in STYLES:
        hist_path = results_dir / f"ppo_{town}_{style}_training_history.json"
        if hist_path.exists():
            with open(hist_path) as f:
                hist = json.load(f)
            ax.plot(hist["entropy_bonuses"], label=style, color=STYLE_COLORS[style], alpha=0.8)
    ax.set_title(town)
    ax.set_ylabel("Entropy")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Entropy Curves -- All Styles per Town", fontsize=14)
plt.tight_layout()
plt.show()

## 3.8 Save

### 3.8.1 Checkpoint Inventory

After all training runs complete, we verify that the expected checkpoint files exist on disk. Each town-style combination should produce one `ppo_{town}_{style}_best.pt` file. Missing checkpoints indicate that training did not reach the minimum reward threshold for that combination, or that CARLA crashed during training. The file sizes are printed as a sanity check -- all checkpoints should be roughly the same size since they share the same ActorCritic architecture.

In [ ]:
models_dir = PROJECT_ROOT / "models"

print("=== PPO Checkpoint Inventory ===")
print(f"{'Checkpoint':<45s} {'Size (MB)':>10s} {'Status':>8s}")
print("-" * 65)

found = 0
expected = 0
for town in TOWNS:
    for style in STYLES:
        expected += 1
        ckpt = models_dir / f"ppo_{town}_{style}_best.pt"
        if ckpt.exists():
            size_mb = ckpt.stat().st_size / 1024 / 1024
            print(f"{ckpt.name:<45s} {size_mb:>9.1f}M {'OK':>8s}")
            found += 1
        else:
            print(f"{ckpt.name:<45s} {'---':>10s} {'MISSING':>8s}")

print(f"\nFound {found}/{expected} checkpoints.")

### 3.8.2 Config Snapshot

We save a JSON snapshot of the PPO configuration that was active during this notebook run. This creates a permanent record alongside the training histories so that future analysis can always trace which hyperparameters produced each set of checkpoints. The snapshot is written to `results/ppo_config.json` and includes all values from `configs/ppo.yaml` plus the reward profiles.

In [ ]:
config_snapshot_path = results_dir / "ppo_config.json"
config_snapshot_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_snapshot_path, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Config snapshot saved to: {config_snapshot_path}")
print(f"File size: {config_snapshot_path.stat().st_size} bytes")
print("\nNotebook 03 complete. Proceed to Notebook 04 for evaluation.")